# 05 - Recognition and Appropriateness Rubric

Weighted target-specific SGD logistic models convert scaled handcrafted features to probabilities with low/high uncertainty thresholds. Every rule is evaluated after a failure so all reasons remain visible.

In [ ]:
import csv,sys
from pathlib import Path
import pandas as pd
cwd=Path.cwd().resolve(); REPO=next((p for p in [cwd,*cwd.parents] if (p/'implementation'/'src').exists()),None)
if REPO is None: raise RuntimeError('Start Jupyter inside the repository.')
sys.path.insert(0,str(REPO/'implementation'/'src'))
from ipcv_attire.dataset_policy import load_policy,require_showcase_approval
from ipcv_attire.pipeline import AttirePipeline
from ipcv_attire.training import train_pipeline
DATA=REPO/'implementation'/'data'; POLICY=load_policy(DATA/'dataset-policy.json'); BUNDLE=REPO/'implementation'/'models'/'classical-attire-smoke'
if not (BUNDLE/'manifest.json').exists(): train_pipeline(manifest_dir=DATA/'interim'/'manifests',fashionpedia_root=DATA/'raw'/'fashionpedia',policy=POLICY,bundle_dir=BUNDLE,profile_name='smoke')
pipeline=AttirePipeline.load(BUNDLE)


## Rubric

Immediate failures: revealing or round-neck casual tops, casual/tight or damaged bottoms, leisurewear, and headwear. Required evidence: collar, allowed sleeve, formal-bottom proxy, and footwear. Suspected skorts require review. Tucked-in status, piercing count, exact open-toe subtype, and customary-headgear exceptions are unsupported.

In [ ]:
definitions=POLICY['compliance_rules']['rule_definitions']; display(pd.DataFrame([{'rule_id':key,**value} for key,value in definitions.items()]).set_index('rule_id')); print('Unsupported:',POLICY['compliance_rules']['unsupported_conditions']); print('Proxies:',POLICY['compliance_rules']['proxies'])


In [ ]:
DEMO_IMAGE_ID='13665'
with (DATA/'interim'/'manifests'/'fashionpedia-images.csv').open(encoding='utf-8',newline='') as handle: record=next(r for r in csv.DictReader(handle) if r['image_id']==DEMO_IMAGE_ID)
require_showcase_approval(DEMO_IMAGE_ID,DATA/'showcase-manifest.csv',display_risk=record['display_risk']=='1'); report=pipeline.analyze(DATA/'raw'/'fashionpedia'/record['relative_image_path'])
predictions=[{'outfit':o.outfit_id,'component':c.component_id,**p.__dict__} for o in report.outfits for c in o.components for p in c.predictions]; display(pd.DataFrame(predictions))


In [ ]:
rules=[{'outfit':o.outfit_id,'decision':o.decision.value,'rule':r.rule_id,'status':r.status.value,'confidence':r.confidence,'evidence':r.evidence_region,'reason':r.reason} for o in report.outfits for r in o.rules]; display(pd.DataFrame(rules)); print('IMAGE DECISION:',report.to_dict()['user_facing_decision'].upper())
for o in report.outfits: print(o.outfit_id,'passed=',o.passed_reasons,'failed=',o.failed_reasons,'review=',o.review_reasons,'unsupported=',o.unsupported_reasons)


## Precedence

Any confident failure makes an outfit inappropriate. Otherwise incomplete required evidence means review-required. Appropriate requires all supported requirements and no prohibition. Across outfits, failure wins, then review, then appropriate.